# Olympic Athletes: 120 Years of History

## 1. Dataset Selection
**Dataset:** 120 years of Olympic history: athletes and results



**Source:** Kaggle (heesoo37), originally scraped from sports-reference.com. Covers Athens 1896 to Rio 2016.



**Why I chose it:** I'm interested in sport, and the data is realistically messy. Height, weight and age are missing for many athletes mainly more towards the earlier records, and medal is null for everyone who didn't win one. That makes it a good test of cleaning decisions.

**Questions to explore:**
1. How has female participation changed over time?
2. How have athletes' height and weight changed over time, and do they differ by sport?
3. Which countries have won the most medals, and does this change by era?
4. How does athlete age vary by sport?
5. Do host countries win more medals than usual (home advantage)?

### 2. Initial Data Exploration and Understanding

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

athletes = pd.read_csv("../data/raw/athlete_events.csv")
regions = pd.read_csv("../data/raw/noc_regions.csv")

print("Athletes:", athletes.shape)
print("Regions:", regions.shape)

Athletes: (271116, 15)
Regions: (230, 3)


In [6]:
athletes.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


In [7]:
print("Columns:", list(athletes.columns))
athletes.info()

Columns: ['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight', 'Team', 'NOC', 'Games', 'Year', 'Season', 'City', 'Sport', 'Event', 'Medal']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 15 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  object 
 2   Sex     271116 non-null  object 
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  object 
 7   NOC     271116 non-null  object 
 8   Games   271116 non-null  object 
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  object 
 11  City    271116 non-null  object 
 12  Sport   271116 non-null  object 
 13  Event   271116 non-null  object 
 14  Medal   39783 non-null   object 
dtypes: float64(3), int64(2), object(10)
memory usage: 31.0+ MB


In [12]:
missing = pd.DataFrame({
    "missing": athletes.isna().sum(),
    "percent": (athletes.isna().mean() * 100).round(2)
}).sort_values("percent", ascending=False)
missing

,missing,percent
Medal,231333,85.33
Weight,62875,23.19
Height,60171,22.19
Age,9474,3.49
Sex,0,0.00
ID,0,0.00
Name,0,0.00
Team,0,0.00
NOC,0,0.00
Year,0,0.00


In [13]:
display(athletes.describe().round(2))
print("Duplicate rows:", athletes.duplicated().sum())
print("Years:", athletes["Year"].min(), "to", athletes["Year"].max())
print("Seasons:", athletes["Season"].unique())

,ID,Age,Height,Weight,Year
count,271116.00,261642.00,210945.00,208241.00,271116.00
mean,68248.95,25.56,175.34,70.70,1978.38
std,39022.29,6.39,10.52,14.35,29.88
min,1.00,10.00,127.00,25.00,1896.00
25%,34643.00,21.00,168.00,60.00,1960.00
50%,68205.00,24.00,175.00,70.00,1988.00
75%,102097.25,28.00,183.00,79.00,2002.00
max,135571.00,97.00,226.00,214.00,2016.00


Duplicate rows: 1385
Years: 1896 to 2016
Seasons: ['Summer' 'Winter']


In [14]:
regions.head()

,NOC,region,notes
0,AFG,Afghanistan,NaN
1,AHO,Curacao,Netherlands Antilles
2,ALB,Albania,NaN
3,ALG,Algeria,NaN
4,AND,Andorra,NaN


## 3. Data Cleaning

### Decision log
| Issue | Finding | Decision | Reason |
|---|---|---|---|
| Duplicate rows | 1,385 exact duplicates | Drop them | Same athlete, same event, same Games recorded twice. They would inflate counts and medal totals. |
| Medal nulls (85%) | Null means no medal | Recode to "No Medal" and add a `Won_Medal` flag | These are not missing data. Leaving them as NaN would hide non-medallists from groupby counts. |
| Height / Weight / Age missing | 22% / 23% / 3.5% | Do not impute. Leave as NaN and drop per-question only | Missingness is concentrated in older Games, so filling with averages would invent modern-looking athletes in 1900. Dropping per question keeps the rest of each row usable. |
| Art Competitions | Non-athletic events with odd ages and no body measurements | Remove | Not physical sport, and they distort age and physique analysis. |
| Country names | `NOC` codes only | Merge `noc_regions.csv` to get country names, fall back to `Team` where there is no match | Readable charts. |
| Data types | Text columns stored as object, Age as float | Convert to category / nullable integer | Less memory, clearer intent. |

In [15]:
df = athletes.copy()   # duplicated incase i mess anything up and would like a extra version and is good for comprison

# Dups
print("Rows before:", len(df))
df = df.drop_duplicates()
print("Rows after: ", len(df), "| removed:", len(athletes) - len(df))

# Same athlete listed twice for the same event at the same Games 
print("Same athlete/Games/Event repeats:", df.duplicated(subset=["ID", "Games", "Event"]).sum())

# Art Competitions
art = df["Sport"] == "Art Competitions"
print("Art Competitions rows:", art.sum(), "| years:", df.loc[art, "Year"].min(), "-", df.loc[art, "Year"].max())
df = df[~art]
print("Rows now:", len(df))

Rows before: 271116
Rows after:  269731 | removed: 1385
Same athlete/Games/Event repeats: 95
Art Competitions rows: 2263 | years: 1912 - 1948
Rows now: 267468
